In [1]:
def spr_interp(data_rcm_grid, data_obs_grid, n_spatterns=10):
    """
    Vectorized version of the spatial pattern regression (SPR) interpolation using PCA-based spatial patterns
    derived from RCM data to interpolate missing observation data.

    Parameters:
    - data_rcm_grid: numpy array of shape (n_days, n_stations)
        RCM data used to extract spatial patterns.
    - data_obs_grid: numpy array of shape (m_days, n_stations)
        Observation data with potential missing values (NaNs).
    - n_spatterns: int, default=10
        Number of spatial patterns (PCA components) to extract and use.

    Returns:
    - data_interp_spr: numpy array of shape (m_days, n_stations)
        Interpolated observation data.
    """

    # Check for dimension consistency
    if data_rcm_grid.shape[1] != data_obs_grid.shape[1]:
        raise ValueError("RCM and OBS grids must have the same number of stations (columns).")
    if n_spatterns > data_rcm_grid.shape[1]:
        raise ValueError("Number of spatial patterns cannot exceed number of stations.")

    # Step 1: Compute spatial patterns using PCA (TruncatedSVD)
    scaler = StandardScaler(with_mean=True, with_std=False)
    data_rcm_scaled = scaler.fit_transform(data_rcm_grid)
    centres_pca = scaler.mean_

    svd_rcm = TruncatedSVD(n_components=n_spatterns)
    svd_rcm.fit(data_rcm_scaled)
    spatterns = svd_rcm.components_.T  # shape: (n_stations, n_spatterns)

    # Step 2: Remove stations (columns) with NaNs in the first day of obs data
    first_day = data_obs_grid[0, :]
    indexes_NA_col = np.where(np.isnan(first_day))[0]
    data_obs_valid = np.delete(data_obs_grid, indexes_NA_col, axis=1)
    spatterns_model = np.delete(spatterns, indexes_NA_col, axis=0)
    centres_model = np.delete(centres_pca, indexes_NA_col)

    # Step 3: Center all observation data at once
    data_obs_centered = data_obs_valid - centres_model

    # Step 4: Solve the least squares problem for all days at once
    pattern_weights = np.linalg.lstsq(spatterns_model, data_obs_centered.T, rcond=None)[0]

    # Step 5: Reconstruct the data for all days at once
    reconstructed_data = spatterns @ pattern_weights + centres_pca.reshape(-1, 1)

    # Step 6: Transpose the result to match original shape (n_days, n_stations)
    data_interp_spr = reconstructed_data.T

    return data_interp_spr
